In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from pysheds.grid import Grid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import shape
from shapely.ops import nearest_points
from rasterio.features import shapes
from tqdm import tqdm

In [ ]:
watershed = gpd.read_file("GIS/WATERSHEDS.geojson")
watershed = watershed[watershed['NAME'] == 'Popes Head Creek']
watershed = watershed.to_crs('EPSG:26918')

In [ ]:
# load elevation data for particular watershed
with rasterio.open("elevation/fairfax_dem_1m.tif") as src:    
    out_image, out_transform = mask(src, watershed.geometry, crop=True)
    out_meta = src.meta.copy()
    out_meta.update({
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform
    })


clipped_dem_path = "elevation/popes_head_dem.tif"
with rasterio.open(clipped_dem_path, 'w', **out_meta) as dest:
    dest.write(out_image)

In [ ]:
grid = Grid.from_raster(clipped_dem_path)
dem = grid.read_raster(clipped_dem_path)

In [ ]:
plt.imshow(np.where(dem == -9999, np.nan, dem), cmap='terrain')
plt.title('Popes Head Creek DEM')
plt.colorbar(label='Elevation (m)')
plt.axis('off')
plt.show()

In [ ]:
pit_filled = grid.fill_pits(dem)
flooded = grid.fill_depressions(pit_filled)
inflated = grid.resolve_flats(flooded)
flowdir = grid.flowdir(inflated)
acc = grid.accumulation(flowdir)
acc = np.where(dem == -9999, np.nan, acc)

In [ ]:
from scipy.ndimage import gaussian_filter
acc_smooth = np.log10(acc+1)
acc_smooth = gaussian_filter(acc_smooth, sigma=3)
plt.imshow(acc_smooth)
plt.axis('off')
plt.colorbar(label='log2(Accumulation + 1)')
plt.title('Popes Head Creek Smoothed Accumulation')
plt.show()

In [ ]:
plt.imshow(np.where(dem == -9999, np.nan, flowdir))
plt.axis('off')
plt.colorbar(label='Flow Direction')
plt.show()

In [ ]:
from scipy.ndimage import binary_fill_holes
subcatchments = []
mask = np.where(dem == -9999, True, False)
thresh = np.percentile(acc.flatten()[~np.isnan(acc.flatten())], 99.9) # controls how much pooling should occur to qualify
max_acc = np.nanmax(acc[~mask])
nan_area = np.sum(mask)
area = np.sum(~mask)
while max_acc > thresh:
    i, j = [(i, j) for i, j in np.argwhere(acc == max_acc) if not mask[i, j]][0]
    sub = grid.catchment(x=j, y=i, fdir=flowdir, xytype='index')
    sub = np.where(mask, False, sub)
    mask[i, j] = True
    if not sub.sum():
        max_acc = np.nanmax(acc[~mask])
        continue
    sub = binary_fill_holes(sub)
    subcatchment_area += sub.sum()
    subcatchments.append(sub)
    mask = np.logical_or(mask, sub)
    print(f"{(mask.sum()-nan_area)/area*100:.2f}%")
    max_acc = np.nanmax(acc[~mask])
    

In [ ]:
from scipy.ndimage import distance_transform_edt

labels = np.zeros_like(acc)
for i, sub in enumerate(subcatchments):
    labels[sub] = i+1
unassigned = labels == 0

dist, (inds_y, inds_x) = distance_transform_edt(
    unassigned,  
    return_indices=True
)

In [ ]:
nearest_labels = labels[inds_y, inds_x]

In [ ]:
labels_filled = labels.copy()
labels_filled[unassigned] = nearest_labels[unassigned]
labels_filled = np.where(dem == -9999, np.nan, labels_filled)

In [ ]:
plt.imshow(labels_filled)
plt.axis('off')
plt.colorbar(label='Subcatchment ID')
plt.show()

In [ ]:
# for every value in labels filled
# if this value is surrounded by a value that it is not equal to on all sides
# set the value equal to one of the surrounding values
for r in tqdm(range(labels.shape[0])):
    for c in range(labels.shape[1]):
        if r == 0 or c == 0 or r == labels.shape[0] - 1 or c == labels.shape[1] - 1:
            continue
        if np.isnan(labels_filled[r, c]):
            continue
        if (
            labels_filled[r-1, c] != labels_filled[r, c] and
            labels_filled[r+1, c] != labels_filled[r, c] and
            labels_filled[r, c-1] != labels_filled[r, c] and
            labels_filled[r, c+1] != labels_filled[r, c]
        ):
            labels_filled[r, c] = max(
                labels_filled[r-1, c],
                labels_filled[r+1, c],
                labels_filled[r, c-1],
                labels_filled[r, c+1]
            )

In [ ]:
filled_subcatchments = [
    labels_filled == i
    for i in range(1, len(subcatchments) + 1)
]

In [ ]:
polygons = []
ids = []
for i, mask in tqdm(list(enumerate(filled_subcatchments, start=1))):
    arr = mask.astype(np.uint8)

    for geom, val in shapes(arr, mask=arr==1, transform=grid.affine):
        polygons.append(shape(geom))
        ids.append(i)

In [ ]:
gdf = gpd.GeoDataFrame(
    {"subcatchment_id": ids, "geometry": polygons},
    crs=grid.crs
)

In [ ]:
gdf.to_file("subcatchments.geojson", driver="GeoJSON")

## Analyze pipe system

In [ ]:
pipes = gpd.read_file('stormnet/14.geojson')

In [ ]:
pipes.to_crs(watershed.crs, inplace=True)
pipes = gpd.clip(pipes, watershed)
pipes = pipes[((pipes.TYPE == 'Pipe') | (pipes.TYPE == 'Culvert'))]
pipes = pipes[pipes.geometry.geom_type == 'LineString']

### Get nodes for each pipe

In [ ]:
node_records = []
    
for idx, row in pipes.iterrows():
    line = row.geometry
    start_point = line.coords[0]
    end_point = line.coords[-1]

    node_records.append({
        "pipe_id": row.get("STORMNET_ID", idx),
        "node_type": "start",
        "geometry": gpd.points_from_xy([start_point[0]], [start_point[1]])[0]
    })

    node_records.append({
        "pipe_id": row.get("STORMNET_ID", idx),
        "node_type": "end",
        "geometry": gpd.points_from_xy([end_point[0]], [end_point[1]])[0]
    })
raw_nodes = gpd.GeoDataFrame(node_records, crs=pipes.crs)
raw_nodes.head()

### Merge node junctions (deduplication)

In [ ]:
nodes_buffer = raw_nodes.copy()
nodes_buffer['geometry'] = nodes_buffer.geometry.buffer(0.75)  # 0.75m merge threshold

# dissolve overlapping buffers
merged = nodes_buffer.dissolve().explode(index_parts=False)

# create representative node as centroid
unique_nodes = merged.copy()
unique_nodes['geometry'] = merged.centroid
unique_nodes = unique_nodes.reset_index(drop=True)

# assign node IDs
unique_nodes['node_id'] = ["J{:05d}".format(i) for i in range(len(unique_nodes))]
unique_nodes.head()

### Find DEM elevation at each node

In [ ]:
dem = rasterio.open("elevation/popes_head_dem.tif")

In [ ]:
elevations = []
for geom in tqdm(unique_nodes.geometry):
    x, y = geom.x, geom.y
    row, col = dem.index(x, y)
    elevations.append(dem.read(1)[row, col])

unique_nodes['elevation'] = elevations

### Link nodes back to pipes

In [ ]:
def get_nearest_node(point, nodes):
    # returns node_id of nearest node
    point = gpd.points_from_xy([point[0]], [point[1]])[0]
    distances = nodes.distance(point)
    return nodes.loc[distances.idxmin(), "node_id"]

pipes["up_node"] = pipes.geometry.apply(lambda g: get_nearest_node(g.coords[0], unique_nodes))
pipes["down_node"] = pipes.geometry.apply(lambda g: get_nearest_node(g.coords[-1], unique_nodes))

### Compute slope and invert elevations

In [ ]:
pipes["length_calc"] = pipes.geometry.length

pipes = pipes.merge(unique_nodes[["node_id", "elevation"]], 
                    left_on="up_node", right_on="node_id", how="left")
pipes = pipes.rename(columns={"elevation": "up_elev"})

pipes = pipes.merge(unique_nodes[["node_id", "elevation"]],
                    left_on="down_node", right_on="node_id", how="left")
pipes = pipes.rename(columns={"elevation": "down_elev"})

pipes["slope_calc"] = (pipes["up_elev"] - pipes["down_elev"]) / pipes["length_calc"]

# use existing slope if valid, else DEM-derived slope
pipes["slope_final"] = pipes.apply(
    lambda r: r["SLOPE"] if (pd.notnull(r["SLOPE"]) and r["SLOPE"] > 0) else r["slope_calc"], 
    axis=1
)
# constrain slopes between 0 and 10
pipes["SLOPE"] = pipes["slope_final"].clip(0, 10)

In [ ]:
# estimate invert elevations
cover_depth = 1.0

pipes["inv_up"] = pipes["up_elev"] - cover_depth
pipes["inv_down"] = pipes["down_elev"] - cover_depth

In [ ]:
lines = []
max_depth = 3
lines.append("[JUNCTIONS]")
lines.append(";;ID           Elevation      MaxDepth")

for _, row in unique_nodes.iterrows():
    line = f"{row['node_id']:15s} {row['elevation']:10.3f} {max_depth:10.2f}"
    lines.append(line)

lines.append("")  # blank line after section
JUNCTIONS = "\n".join(lines)

In [ ]:
cow = gpd.read_file('stormnet/13.geojson')

In [ ]:
JUNCTIONS